# Classification binaire d'images

Notebook de suivi du projet : TP1 CNN scratch, TP2 augmentation + Dropout, TP3 transfer learning MobileNetV2.

## Phase 1.1 - Setup Colab et organisation du dataset

Objectif : configurer Colab, telecharger le dataset Kaggle, organiser les images en `train/val` par classe, puis afficher quelques exemples pour verifier le split.

In [ ]:
# === CONFIGURATION DU PROJET ===
# Dataset Kaggle : https://www.kaggle.com/datasets/chetankv/dogs-cats-images
# Ces constantes sont le seul endroit a modifier pour changer de sujet.

CLASS_A = "cat"
CLASS_B = "dog"

DATA_ROOT = "/content/data"
RAW_ROOT = f"{DATA_ROOT}/raw"
KAGGLE_DATASET = "chetankv/dogs-cats-images"
KAGGLE_ZIP = "dogs-cats-images.zip"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED = 42
TRAIN_RATIO = 0.80

### 1.1.1 - Installation et authentification Kaggle

Dans Colab, telecharger `kaggle.json` depuis Kaggle : Settings > API > Create New Token.

In [ ]:
!pip install kaggle -q

In [ ]:
import os
from google.colab import files

uploaded = files.upload()  # selectionner kaggle.json

if "kaggle.json" not in uploaded:
    raise FileNotFoundError("Le fichier kaggle.json doit etre selectionne.")

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
if os.path.exists(kaggle_json_path):
    os.remove(kaggle_json_path)

os.rename("kaggle.json", kaggle_json_path)
os.chmod(kaggle_json_path, 0o600)

print(f"Token Kaggle installe : {kaggle_json_path}")
!ls -la ~/.kaggle/

### 1.1.2 - Telechargement et extraction du dataset

In [ ]:
import os

os.makedirs(RAW_ROOT, exist_ok=True)
!kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_ROOT}
!unzip -q -o {RAW_ROOT}/{KAGGLE_ZIP} -d {RAW_ROOT}

print("Extraction terminee dans", RAW_ROOT)

In [ ]:
from pathlib import Path

for path in sorted(Path(RAW_ROOT).glob("**/*"))[:40]:
    print(path.relative_to(RAW_ROOT))

### 1.1.3 - Organisation en train/val

La structure finale attendue est `DATA_ROOT/train/cat`, `DATA_ROOT/train/dog`, `DATA_ROOT/val/cat`, `DATA_ROOT/val/dog`.

In [ ]:
import random
import shutil
from pathlib import Path


def is_image(path):
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def collect_images_for_class(raw_root, class_name):
    raw_root = Path(raw_root)
    singular = class_name.lower()
    plural = f"{singular}s"

    class_dirs = [
        path for path in raw_root.glob("**/*")
        if path.is_dir() and path.name.lower() in {singular, plural}
    ]

    files = []
    for class_dir in class_dirs:
        files.extend(path for path in class_dir.rglob("*") if is_image(path))

    unique_files = sorted(set(files))
    if not unique_files:
        raise ValueError(f"Aucune image trouvee pour la classe {class_name} dans {raw_root}")

    return unique_files


files_a = collect_images_for_class(RAW_ROOT, CLASS_A)
files_b = collect_images_for_class(RAW_ROOT, CLASS_B)

print(CLASS_A, len(files_a), "images brutes")
print(CLASS_B, len(files_b), "images brutes")

In [ ]:
import os
import random
import shutil


def reset_output_dirs(data_root, classes):
    for split in ["train", "val"]:
        for cls in classes:
            path = os.path.join(data_root, split, cls)
            if os.path.exists(path):
                shutil.rmtree(path)
            os.makedirs(path, exist_ok=True)


def split_files(files, seed=SEED, train_ratio=TRAIN_RATIO):
    files = list(files)
    rng = random.Random(seed)
    rng.shuffle(files)
    split_index = int(len(files) * train_ratio)
    return files[:split_index], files[split_index:]


def copy_split(files, split, class_name):
    target_dir = os.path.join(DATA_ROOT, split, class_name)
    for source_path in files:
        target_path = os.path.join(target_dir, source_path.name)
        if os.path.exists(target_path):
            stem, suffix = source_path.stem, source_path.suffix
            target_path = os.path.join(target_dir, f"{stem}_{abs(hash(str(source_path))) % 10**8}{suffix}")
        shutil.copy2(source_path, target_path)


reset_output_dirs(DATA_ROOT, [CLASS_A, CLASS_B])

train_a, val_a = split_files(files_a)
train_b, val_b = split_files(files_b)

copy_split(train_a, "train", CLASS_A)
copy_split(val_a, "val", CLASS_A)
copy_split(train_b, "train", CLASS_B)
copy_split(val_b, "val", CLASS_B)

print("Organisation terminee.")

### 1.1.4 - Verification des comptes

In [ ]:
counts = {}

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        path = os.path.join(DATA_ROOT, split, cls)
        image_count = len([name for name in os.listdir(path) if Path(name).suffix.lower() in IMAGE_EXTENSIONS])
        counts[(split, cls)] = image_count
        print(f"{path} : {image_count} images")

for cls in [CLASS_A, CLASS_B]:
    total = counts[("train", cls)] + counts[("val", cls)]
    train_ratio = counts[("train", cls)] / total
    print(f"{cls} : ratio train={train_ratio:.2%}, total={total}")
    assert counts[("train", cls)] >= 500, f"Pas assez d'images train pour {cls}"
    assert abs(train_ratio - TRAIN_RATIO) <= 0.01, f"Ratio train/val incorrect pour {cls}"

print("Verification des comptes OK.")

### 1.1.5 - Verification visuelle

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt


def sample_images(split, class_name, n=3):
    class_dir = Path(DATA_ROOT) / split / class_name
    images = sorted(path for path in class_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    return images[:n]


samples = [(CLASS_A, path) for path in sample_images("train", CLASS_A, 3)]
samples += [(CLASS_B, path) for path in sample_images("train", CLASS_B, 3)]

plt.figure(figsize=(12, 6))
for i, (class_name, image_path) in enumerate(samples):
    image = mpimg.imread(image_path)
    print(f"{image_path.name} | {class_name} | shape={image.shape}")

    if len(image.shape) == 3:
        assert image.shape[-1] in [1, 3, 4], f"Canaux inattendus pour {image_path}: {image.shape}"
    else:
        assert len(image.shape) == 2, f"Shape inattendue pour {image_path}: {image.shape}"

    plt.subplot(2, 3, i + 1)
    plt.imshow(image, cmap="gray" if len(image.shape) == 2 else None)
    plt.title(class_name)
    plt.axis("off")

plt.tight_layout()
plt.show()

### 1.1.6 - Edge case : seed different, ratio stable

Changer le seed modifie les fichiers selectionnes, mais pas les comptes par split sauf effet d'arrondi.

In [ ]:
def split_count_summary(files, seed):
    train_files, val_files = split_files(files, seed=seed)
    return len(train_files), len(val_files)


for cls, files in [(CLASS_A, files_a), (CLASS_B, files_b)]:
    train_42, val_42 = split_count_summary(files, seed=42)
    train_0, val_0 = split_count_summary(files, seed=0)
    print(f"{cls} | seed=42 : train={train_42}, val={val_42}")
    print(f"{cls} | seed=0  : train={train_0}, val={val_0}")
    assert abs(train_42 - train_0) <= 1
    assert abs(val_42 - val_0) <= 1

print("Verification seed OK.")

## TP1 - CNN scratch

A completer pendant la phase 1.2.

## TP2 - Augmentation + Dropout

A completer pendant la phase 2.

## TP3 - Transfer learning MobileNetV2

A completer pendant la phase 3.

## 3.4 - Tableau comparatif

A completer avec les metriques finales des trois modeles.

## 3.5 - Export ou exploration

A completer avec l'export `.tflite` et/ou une direction d'exploration.